In [1]:

from huggingface_hub import notebook_login

notebook_login()


In [17]:
%pip install -q -U torch transformers accelerate huggingface_hub optimum-intel neural-compressor datasets
# https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct

from transformers import AutoTokenizer
import torch
from importlib.metadata import version

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)

print("Device :", DEVICE)
print("Dtype  :", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)
print("Tokenizer chargé.")
print("optimum :", version("optimum"))
print("neural-compressor :", version("neural-compressor"))
print("optimum-intel :", version("optimum-intel"))




[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Device : cpu
Dtype  : torch.float32
Tokenizer chargé.
optimum : 2.1.0
neural-compressor : 3.10
optimum-intel : 1.27.0
neural-compressor : 3.10


In [21]:
# https://huggingface.co/docs/optimum/v1.2.1/en/intel/pruning

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from optimum.intel.intel import INCTrainer, IncPruner

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
OUTPUT_DIR = "./smollm2-pruned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

# Petit corpus d'exemple
dataset = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    split="train[:1%]",
)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names,
)
train_dataset = train_dataset.filter(
    lambda example: len(example["input_ids"]) > 1
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    save_strategy="no",
    report_to="none",
)

trainer = INCTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    ),
    tokenizer=tokenizer,
)

pruner = IncPruner.from_config(
    model_name_or_path=MODEL_ID,
    inc_config="./prune.yml",
    config_name="prune.yml",
)

trainer.train(pruner=pruner)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)




ModuleNotFoundError: No module named 'optimum.intel.intel'

In [ ]:
PATH = "./models/pruned_model"

model_quantized.save_pretrained(
    PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(
    PATH
)

print(
    f"Modèle sauvegardé dans : {PATH}"
)